```
**Data modeling** = deciding how to structure data so it can be:

Stored efficiently

Queried correctly

Understood by humans

Maintained over time
```

**Normalization and De-Normalization are database technique to organize data**

## 1. What is normalization?

Normalization means splitting data into multiple related tables to reduce duplicate data and improve data consistency.

Imagine you have this table:

**Before normalization**

| Order_ID | Customer_Name | Customer_City | Product |
|---|---|---|---|
| 101 | Faiz | Hyderabad | Laptop |
| 102 | Faiz | Hyderabad | Mouse |
| 103 | Rahul | Mumbai | Keyboard |

Notice that Faiz's name and city are repeated. If Faiz changes cities, you might need to update multiple rows.

We can split this into separate tables.

**Customers table**

| Customer_ID | Customer_Name | Customer_City |
|---|---|---|
| 1 | Faiz | Hyderabad |
| 2 | Rahul | Mumbai |

**Orders table**

| Order_ID | Customer_ID | Product |
|---|---|---|
| 101 | 1 | Laptop |
| 102 | 1 | Mouse |
| 103 | 2 | Keyboard |

Now customer information is stored once, and orders reference the customer using `Customer_ID`.

Benefits:

- Reduces duplicate data.
- Makes updates more consistent.
- Helps prevent insertion, update, and deletion anomalies.

## 2. What is denormalization?

Denormalization means combining tables or intentionally duplicating data to make data retrieval simpler or faster.

Suppose you frequently need order details along with customer names and cities. Instead of joining the Customers and Orders tables every time, you could store the combined result in one table.

**Denormalized table**

| Order_ID | Customer_Name | Customer_City | Product |
|---|---|---|---|
| 101 | Faiz | Hyderabad | Laptop |
| 102 | Faiz | Hyderabad | Mouse |
| 103 | Rahul | Mumbai | Keyboard |

This is easier to query because customer details are already included.

Benefits:

- Can reduce the need for joins.
- Can improve read performance for particular workloads.
- Is useful for reporting and analytics.

## 1. First Normal Form (1NF)

A table is in 1NF when:
- Each cell contains a single value.
- There are no repeating groups or multiple values in one cell.
- Each row can be uniquely identified.

**Example: Before 1NF**

| Student_ID | Student_Name | Subjects |
|---|---|---|
| 1 | Faiz | SQL, Python |
| 2 | Rahul | Java, Spark |

The `Subjects` column contains multiple values in a single cell.

**After 1NF**

| Student_ID | Student_Name | Subject |
|---|---|---|
| 1 | Faiz | SQL |
| 1 | Faiz | Python |
| 2 | Rahul | Java |
| 2 | Rahul | Spark |

Now each cell contains a single value.

## 2. Second Normal Form (2NF)

Every non-key column must depend on the whole primary key, not just part of it.

**Example: Before 2NF**

| Student_ID | Course_ID | Student_Name | Course_Name |
|---|---|---|---|
| 1 | C101 | Faiz | SQL |
| 1 | C102 | Faiz | Python |
| 2 | C101 | Rahul | SQL |

Assume the composite primary key is `(Student_ID, Course_ID)`.

Here:
- `Student_Name` depends only on `Student_ID`.
- `Course_Name` depends only on `Course_ID`.

These are partial dependencies, so the table violates 2NF.

**After 2NF**

**Students table**

| Student_ID | Student_Name |
|---|---|
| 1 | Faiz |
| 2 | Rahul |

**Courses table**

| Course_ID | Course_Name |
|---|---|
| C101 | SQL |
| C102 | Python |

**Enrollments table**

| Student_ID | Course_ID |
|---|---|
| 1 | C101 |
| 1 | C102 |
| 2 | C101 |

Now each non-key attribute depends on the entire key of its table.

## 3. Third Normal Form (3NF)

A table is in 3NF when:
- It is already in 2NF.
- Non-key columns do not depend on other non-key columns. In other words, there are no transitive dependencies of non-key attributes on a candidate key.

**Example: Before 3NF**

| Employee_ID | Employee_Name | Department_ID | Department_Name |
|---|---|---|---|
| 101 | Faiz | D1 | Engineering |
| 102 | Rahul | D2 | Finance |
| 103 | Ali | D1 | Engineering |

Assume `Employee_ID` is the primary key.

Here:
- `Employee_Name` depends on `Employee_ID`.
- `Department_ID` depends on `Employee_ID`.
- `Department_Name` depends on `Department_ID`, not directly on `Employee_ID`.

Therefore, there is a transitive dependency:

`Employee_ID → Department_ID → Department_Name`

**After 3NF**

**Employees table**

| Employee_ID | Employee_Name | Department_ID |
|---|---|---|
| 101 | Faiz | D1 |
| 102 | Rahul | D2 |
| 103 | Ali | D1 |

**Departments table**

| Department_ID | Department_Name |
|---|---|
| D1 | Engineering |
| D2 | Finance |

Now department information is stored separately, eliminating the transitive dependency.

## Interview Summary

| Normal Form | Main Rule |
|---|---|
| 1NF | Each cell contains a single value; no repeating groups. |
| 2NF | 1NF + no partial dependency on a composite candidate key. |
| 3NF | 2NF + no transitive dependency of non-key attributes on a candidate key. |

**Easy way to remember:**
- **1NF:** One value per cell.
- **2NF:** Depend on the whole key.
- **3NF:** Depend on the key, not another non-key column.

## Star Schema

### Structure

**One fact table** in the center, surrounded by **flat (denormalized) dimension tables**.

```text
                [dim_date]
                     |
                     |
[dim_product] — [fact_sales] — [dim_customer]
                     |
                     |
                [dim_store]
```

### The Tables

`fact_sales` (the center — measures + foreign keys)

| sale_id | date_key | product_key | customer_key | store_key | quantity | revenue |
| ------- | -------- | ----------- | ------------ | --------- | -------- | ------- |
| 1       | 20240215 | 101         | 201          | 301       | 2        | 250     |
| 2       | 20240215 | 102         | 202          | 301       | 1        | 150     |
| 3       | 20240216 | 101         | 203          | 302       | 3        | 375     |

`dim_product` (flat — everything in one table)

| product_key | product_name | subcategory | category    | brand |
| ----------- | ------------ | ----------- | ----------- | ----- |
| 101         | Widget Pro   | Widgets     | Electronics | Acme  |
| 102         | Gadget Mini  | Gadgets     | Electronics | Beta  |

`dim_customer` (flat)

| customer_key | name  | city   | state | country | segment  |
| ------------ | ----- | ------ | ----- | ------- | -------- |
| 201          | Alice | NYC    | NY    | USA     | Premium  |
| 202          | Bob   | Boston | MA    | USA     | Standard |
| 203          | Carol | Miami  | FL    | USA     | Premium  |

`dim_store` (flat)

| store_key | store_name    | city  | state | region    |
| --------- | ------------- | ----- | ----- | --------- |
| 301       | Downtown NYC  | NYC   | NY    | Northeast |
| 302       | Miami Central | Miami | FL    | Southeast |

`dim_date`

| date_key | date       | day | month | quarter | year |
| -------- | ---------- | --- | ----- | ------- | ---- |
| 20240215 | 2024-02-15 | 15  | 2     | Q1      | 2024 |

### Key Characteristics

| Feature          | Value                                           |
| ---------------- | ----------------------------------------------- |
| Number of tables | 1 fact + 4 dims = **5 tables**                  |
| Dimension shape  | Flat (denormalized)                             |
| Redundancy       | Higher (category repeated in every product row) |
| Joins needed     | **Few**                                         |
| Query complexity | Simple                                          |
| Query speed      | Fast                                            |

### Typical Query

```sql
SELECT
    p.category,
    d.year,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_product p ON f.product_key = p.product_key
JOIN dim_date d ON f.date_key = d.date_key
WHERE d.year = 2024
GROUP BY p.category, d.year;
```

**Only 2 joins.** Clean and fast.

---

## Snowflake Schema

### Structure

**Same fact table**, but dimensions are **normalized** — split into multiple related tables.

```text
                     [dim_date]
                          |
                          |
[dim_product] — [fact_sales] — [dim_customer]
      |                           |
      v                           v
[dim_subcategory]           [dim_city]
      |                           |
      v                           v
[dim_category]              [dim_state]
                                  |
                                  v
                            [dim_country]
```

### The Tables

`fact_sales` — same as Star

`dim_product` (normalized — only immediate attributes)

| product_key | product_name | subcategory_key | brand |
| ----------- | ------------ | --------------- | ----- |
| 101         | Widget Pro   | 501             | Acme  |
| 102         | Gadget Mini  | 502             | Beta  |

`dim_subcategory`

| subcategory_key | subcategory_name | category_key |
| --------------- | ---------------- | ------------ |
| 501             | Widgets          | 601          |
| 502             | Gadgets          | 601          |

`dim_category`

| category_key | category_name |
| ------------ | ------------- |
| 601          | Electronics   |

`dim_customer` (normalized)

| customer_key | name  | city_key | segment  |
| ------------ | ----- | -------- | -------- |
| 201          | Alice | 701      | Premium  |
| 202          | Bob   | 702      | Standard |
| 203          | Carol | 703      | Premium  |

`dim_city`

| city_key | city_name | state_key |
| -------- | --------- | --------- |
| 701      | NYC       | 801       |
| 702      | Boston    | 802       |
| 703      | Miami     | 803       |

`dim_state`

| state_key | state_name | country_key |
| --------- | ---------- | ----------- |
| 801       | NY         | 901         |
| 802       | MA         | 901         |
| 803       | FL         | 901         |

`dim_country`

| country_key | country_name |
| ----------- | ------------ |
| 901         | USA          |

### Key Characteristics

| Feature          | Value                             |
| ---------------- | --------------------------------- |
| Number of tables | 1 fact + ~10 dims = **11 tables** |
| Dimension shape  | Normalized (splits)               |
| Redundancy       | Lower (each value stored once)    |
| Joins needed     | **Many**                          |
| Query complexity | Complex                           |
| Query speed      | Slower (more joins)               |

### Typical Query

```sql
SELECT
    cat.category_name,
    d.year,
    SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_product p
    ON f.product_key = p.product_key
JOIN dim_subcategory sub
    ON p.subcategory_key = sub.subcategory_key
JOIN dim_category cat
    ON sub.category_key = cat.category_key
JOIN dim_date d
    ON f.date_key = d.date_key
WHERE d.year = 2024
GROUP BY cat.category_name, d.year;
```
